# Классическое ML: scikit-learn — когда нейросеть не нужна

**Роль:** Преподаватель по ML  
**Уровень:** средний (Python + Pandas + основы ML)  
**Время прохождения:** ~90–120 минут

---

### Чему вы научитесь

- поймёте, **когда** использовать классические модели, а когда — нейросети;
- освоите **5 алгоритмов**: KNN, Decision Tree, Random Forest, SVM, Logistic Regression;
- научитесь **выбирать модель** под задачу;
- освоите **train/test split, cross-validation, GridSearchCV**;
- визуализируете **границы решений** каждого алгоритма;
- построите **ансамбль** (Random Forest) и поймёте, почему он мощный.

### Принцип

> Вы — автор, не пользователь. Каждая строка видна. Никаких чёрных ящиков.

---

### Почему не нейросеть?

```
Задача                          Лучший выбор
─────────────────────────────── ──────────────────
1000 примеров, 10 признаков    → Классическое ML
1 млн примеров, 224x224 картинка → Нейросеть (CNN)
50 примеров, 5 признаков       → Классическое ML (или статистика)
Текст, 100 тыс. документов     → Нейросеть (Transformer)
Таблица клиентов банка         → Классическое ML (Gradient Boosting)
```

Нейросеть — мощный инструмент, но не единственный. Часто дерево решений лучше!

---

## План урока

| # | Шаг | Что делаем |
|---|-----|------------|
| 1 | Подготовка окружения | Импорт библиотек |
| 2 | Данные: Titanic | Загрузка и подготовка |
| 3 | KNN (k-ближайших соседей) | Интуиция + визуализация |
| 4 | Decision Tree (дерево решений) | Как дерево «думает» |
| 5 | Random Forest (случайный лес) | Ансамбль деревьев |
| 6 | SVM (метод опорных векторов) | Граница решений |
| 7 | Logistic Regression | Линейный классификатор |
| 8 | Сравнение моделей | Таблица + визуализация |
| 9 | Cross-validation и GridSearch | Правильная оценка |
| 10 | Практические задания | 5 заданий |

---

---
## Шаг 1. Подготовка окружения


In [ ]:
# ============================================================
# ШАГ 1: Импорт библиотек
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns

# scikit-learn — главная библиотека классического ML
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (accuracy_score, classification_report, 
                             confusion_matrix, roc_curve, auc)

# Алгоритмы
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression

# Настройки
matplotlib.rcParams['font.sans-serif'] = ['Noto Sans SC', 'DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False
sns.set_theme(style='whitegrid')

import warnings
warnings.filterwarnings('ignore')

print('✅ Все библиотеки импортированы')
print(f'  scikit-learn доступен')

---
## Шаг 2. Данные: Titanic — подготовка

Используем датасет Titanic (из блокнота 05 вы его уже знаете). Быстро подготовим данные.


In [ ]:
# ============================================================
# ШАГ 2: Загрузка и подготовка Titanic
# ============================================================

df = sns.load_dataset('titanic')
print(f'Датасет: {df.shape[0]} строк, {df.shape[1]} столбцов')

# ---- Быстрая предобработка (из блокнота 05) ----
# Удаляем лишние столбцы
df = df.drop(columns=['alive', 'deck', 'who', 'adult_male', 'alone', 'embarked', 'class'], errors='ignore')

# Пропуски
df['age'] = df['age'].fillna(df['age'].median())
df['embark_town'] = df['embark_town'].fillna(df['embark_town'].mode()[0])
df['fare'] = df['fare'].fillna(df['fare'].median())

# Feature engineering
df['family_size'] = df['sibsp'] + df['parch'] + 1
df['is_alone'] = (df['family_size'] == 1).astype(int)
df['sex_encoded'] = df['sex'].map({'male': 0, 'female': 1})
df = pd.get_dummies(df, columns=['embark_town'], prefix='port', drop_first=True)

# Оставляем только числа
df = df.select_dtypes(include=[np.number])

# Отделяем X и y
y = df['survived']
X = df.drop(columns=['survived'])

print(f'Признаки: {X.shape[1]}, Строк: {X.shape[0]}')
print(f'Столбцы: {list(X.columns)}')

# ---- Train/Test split ----
# 80% для обучения, 20% для тестирования
# stratify=y — сохраняем пропорцию классов в обеих частях
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'\nTrain: {X_train.shape[0]} строк')
print(f'Test:  {X_test.shape[0]} строк')
print(f'Доля выживших (train): {y_train.mean():.2%}')
print(f'Доля выживших (test):  {y_test.mean():.2%}')

# ---- Масштабирование ----
# Важно для KNN и SVM (они чувствительны к масштабу!)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)  # обучаем на train
X_test_scaled = scaler.transform(X_test)         # применяем к test
# ВАЖНО: fit только на train! Иначе — утечка данных (data leakage)

print(f'\nМасштабирование: StandardScaler (среднее=0, std=1)')
print(f'До: age=[{X_train["age"].min():.0f}, {X_train["age"].max():.0f}]')
print(f'После: age=[{X_train_scaled[:, 0].min():.2f}, {X_train_scaled[:, 0].max():.2f}]')

---
## Шаг 3. KNN — k-ближайших соседей

### Интуиция

«Скажи мне, кто твои друзья, и я скажу, кто ты.»

KNN — самый простой алгоритм:
1. Смотрим на **k** ближайших объектов (соседей)
2. Каких классов среди них больше — тот и ответ

```
Новый объект → найти k ближайших соседей → голосование → класс
```

**Плюсы:** простой, не требует обучения  
**Минусы:** медленный при больших данных, чувствителен к масштабу


In [ ]:
# ============================================================
# ШАГ 3: KNN — k-ближайших соседей
# ============================================================

# ---- Как k влияет на результат? ----
k_values = range(1, 31)
train_scores = []
test_scores = []

for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_scaled, y_train)
    train_scores.append(knn.score(X_train_scaled, y_train))
    test_scores.append(knn.score(X_test_scaled, y_test))

# Визуализация
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(k_values, train_scores, 'b-o', label='Train accuracy', markersize=4)
ax.plot(k_values, test_scores, 'r-o', label='Test accuracy', markersize=4)
best_k = k_values[np.argmax(test_scores)]
ax.axvline(x=best_k, color='green', linestyle='--', linewidth=2, 
           label=f'Лучший k={best_k}')
ax.set_xlabel('k (количество соседей)')
ax.set_ylabel('Точность')
ax.set_title('KNN: как k влияет на точность', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'Лучший k = {best_k} (test accuracy = {max(test_scores):.2%})')
print()
print('Наблюдения:')
print('  k=1: идеальный train, плохой test (переобучение)')
print('  k=30: и train, и test падают (недообучение)')
print('  Лучший k — компромисс (обычно 5-15)')

# ---- Финальная модель ----
knn_best = KNeighborsClassifier(n_neighbors=best_k)
knn_best.fit(X_train_scaled, y_train)
knn_pred = knn_best.predict(X_test_scaled)
knn_acc = accuracy_score(y_test, knn_pred)
print(f'\nKNN (k={best_k}): accuracy = {knn_acc:.2%}')

---
## Шаг 4. Decision Tree — дерево решений

### Интуиция

Дерево решений = серия вопросов «да/нет»:

```
         Пол = женский?
        /              \
      ДА               НЕТ
     /                  \
  Класс = 1?         Возраст < 10?
  /       \          /           \
ДА         НЕТ     ДА             НЕТ
Выжил!    50/50   Выжил!        Погиб :(
```

**Плюсы:** понятно человеку, не нужно масштабирование, работает с любыми признаками  
**Минусы:** легко переобучается, нестабильно (малое изменение данных → другое дерево)


In [ ]:
# ============================================================
# ШАГ 4: Decision Tree
# ============================================================

# Создаём дерево с ограничением глубины (чтобы не переобучилось)
tree = DecisionTreeClassifier(max_depth=4, random_state=42)
tree.fit(X_train, y_train)  # Дереву НЕ нужно масштабирование!

tree_pred = tree.predict(X_test)
tree_acc = accuracy_score(y_test, tree_pred)
print(f'Decision Tree (depth=4): accuracy = {tree_acc:.2%}')

# ---- Визуализация дерева ----
fig, ax = plt.subplots(figsize=(20, 10))
plot_tree(tree, feature_names=X.columns, class_names=['Погиб', 'Выжил'],
          filled=True, rounded=True, fontsize=9, ax=ax)
ax.set_title('Дерево решений (depth=4)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# ---- Важность признаков ----
# Дерево автоматически определяет, какие признаки важнее!
importance = pd.Series(tree.feature_importances_, index=X.columns)
importance = importance.sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
importance.plot(kind='barh', ax=ax, color='steelblue', edgecolor='white')
ax.set_title('Важность признаков (Decision Tree)', fontsize=13, fontweight='bold')
ax.set_xlabel('Важность')
plt.tight_layout()
plt.show()

print('Самый важный признак:', importance.index[-1])
print('Дерево «решает» в первую очередь по самому информативному признаку!')

# ---- Глубина дерева vs Точность ----
depths = range(1, 21)
train_acc = []
test_acc = []

for d in depths:
    t = DecisionTreeClassifier(max_depth=d, random_state=42)
    t.fit(X_train, y_train)
    train_acc.append(t.score(X_train, y_train))
    test_acc.append(t.score(X_test, y_test))

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(depths, train_acc, 'b-o', label='Train', markersize=4)
ax.plot(depths, test_acc, 'r-o', label='Test', markersize=4)
ax.set_xlabel('Глубина дерева')
ax.set_ylabel('Точность')
ax.set_title('Переобучение дерева: глубина vs точность', fontsize=13, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print('Глубина 1-3: недообучение (мало вопросов)')
print('Глубина 4-6: оптимально')
print('Глубина 10+: переобучение (train=100%, test падает)')

---
## Шаг 5. Random Forest — случайный лес

### Проблема одного дерева

Одно дерево **нестабильно** — малое изменение данных = совсем другое дерево.

### Решение: ансамбль!

Random Forest = **много деревьев** + **голосование**:
1. Обучаем 100 (или 500) разных деревьев
2. Каждое дерево видит **случайную подвыборку** данных
3. Каждое дерево видит **случайную подвыборку** признаков
4. Итоговый ответ = голосование большинства

```
Дерево 1 → Погиб
Дерево 2 → Выжил   ──→ Голосование → Выжил (3 из 5)
Дерево 3 → Выжил
Дерево 4 → Погиб
Дерево 5 → Выжил
```


In [ ]:
# ============================================================
# ШАГ 5: Random Forest
# ============================================================

forest = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42)
forest.fit(X_train, y_train)

forest_pred = forest.predict(X_test)
forest_acc = accuracy_score(y_test, forest_pred)
print(f'Random Forest (100 деревьев, depth=6): accuracy = {forest_acc:.2%}')

# ---- Важность признаков (более надёжная, чем у одного дерева!) ----
forest_importance = pd.Series(forest.feature_importances_, index=X.columns)
forest_importance = forest_importance.sort_values(ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Сравнение: одно дерево vs лес
importance.sort_values(ascending=True).plot(kind='barh', ax=axes[0], color='steelblue')
axes[0].set_title('Важность: одно дерево', fontsize=12, fontweight='bold')

forest_importance.plot(kind='barh', ax=axes[1], color='forestgreen')
axes[1].set_title('Важность: Random Forest', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

print('Лес даёт более стабильную оценку важности признаков!')
print('Топ-3 важных:')
for col in forest_importance.tail(3).index:
    print(f'  {col}: {forest_importance[col]:.3f}')

# ---- Количество деревьев vs Точность ----
n_trees = [1, 5, 10, 20, 50, 100, 200, 500]
oob_scores = []
test_scores_rf = []

for n in n_trees:
    rf = RandomForestClassifier(n_estimators=n, max_depth=6, 
                                 random_state=42, oob_score=True)
    rf.fit(X_train, y_train)
    oob_scores.append(rf.oob_score_)
    test_scores_rf.append(rf.score(X_test, y_test))

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(n_trees, oob_scores, 'b-o', label='OOB score (без теста!)', markersize=6)
ax.plot(n_trees, test_scores_rf, 'r-o', label='Test accuracy', markersize=6)
ax.set_xlabel('Количество деревьев')
ax.set_ylabel('Точность')
ax.set_title('Random Forest: больше деревьев = лучше?', fontsize=13, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print('OOB (Out-of-Bag) — оценка без отдельного теста!')
print('Каждое дерево обучается на ~63% данных, а 37% — «вне мешка»')
print('OOB = средняя точность на «вне-мешковых» примерах')

---
## Шаг 6. SVM — метод опорных векторов

### Интуиция

SVM ищет **лучшую границу** между классами — ту, у которой **максимальный зазор** (margin) до ближайших точек.

```
    ○  ○  ○                    ○  ○  ○
         ○                      ○
  ─────────────── ← граница    ═════════════ ← лучшая граница (макс. зазор)
    ●    ●                      ●        ●
  ●   ●                   ●         ●
```

Ближайшие к границе точки — **опорные векторы** (support vectors).

**Плюсы:** мощный, работает в больших размерностях, есть kernel trick  
**Минусы:** медленный на больших данных, чувствителен к масштабу


In [ ]:
# ============================================================
# ШАГ 6: SVM
# ============================================================

# SVM ЧУВСТВИТЕЛЕН к масштабу — обязательно StandardScaler!
svm = SVC(kernel='rbf', C=1.0, random_state=42)
svm.fit(X_train_scaled, y_train)

svm_pred = svm.predict(X_test_scaled)
svm_acc = accuracy_score(y_test, svm_pred)
print(f'SVM (RBF kernel): accuracy = {svm_acc:.2%}')

# ---- Влияние параметра C ----
# C = штраф за ошибку. Больше C = меньше ошибок на train, но может переобучиться
C_values = [0.01, 0.1, 1.0, 10.0, 100.0]
svm_scores = []

for c in C_values:
    svm_c = SVC(kernel='rbf', C=c, random_state=42)
    svm_c.fit(X_train_scaled, y_train)
    svm_scores.append(svm_c.score(X_test_scaled, y_test))

fig, ax = plt.subplots(figsize=(8, 5))
ax.semilogx(C_values, svm_scores, 'b-o', markersize=8)
ax.set_xlabel('C (штраф за ошибку)')
ax.set_ylabel('Test Accuracy')
ax.set_title('SVM: как параметр C влияет на точность', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

best_C = C_values[np.argmax(svm_scores)]
print(f'Лучший C = {best_C}')
print('Маленький C = широкая граница (недообучение)')
print('Большой C = узкая граница (переобучение)')

---
## Шаг 7. Logistic Regression — линейный классификатор

Несмотря на название — это **классификация**, не регрессия!

### Формула

```
P(класс=1) = sigmoid(w0 + w1*x1 + w2*x2 + ... + wn*xn)

sigmoid(z) = 1 / (1 + exp(-z))
```

Если P > 0.5 → класс 1, иначе → класс 0

**Плюсы:** быстрый, интерпретируемый, выдаёт вероятности  
**Минусы:** только линейная граница решений


In [ ]:
# ============================================================
# ШАГ 7: Logistic Regression
# ============================================================

lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_scaled, y_train)

lr_pred = lr.predict(X_test_scaled)
lr_acc = accuracy_score(y_test, lr_pred)
print(f'Logistic Regression: accuracy = {lr_acc:.2%}')

# ---- Коэффициенты = вклад каждого признака ----
coefs = pd.Series(lr.coef_[0], index=X.columns).sort_values()

fig, ax = plt.subplots(figsize=(10, 6))
colors = ['red' if c < 0 else 'green' for c in coefs]
coefs.plot(kind='barh', ax=ax, color=colors, edgecolor='white')
ax.set_title('Коэффициенты Logistic Regression\n(красный = уменьшает шанс выжить, зелёный = увеличивает)', 
             fontsize=12, fontweight='bold')
ax.set_xlabel('Коэффициент')
ax.axvline(x=0, color='black', linewidth=0.5)
plt.tight_layout()
plt.show()

# ---- Вероятности ----
lr_probs = lr.predict_proba(X_test_scaled[:5])
print('\nВероятности для первых 5 пассажиров:')
for i in range(5):
    p_died = lr_probs[i, 0]
    p_survived = lr_probs[i, 1]
    actual = 'Выжил' if y_test.iloc[i] == 1 else 'Погиб'
    pred = 'Выжил' if lr_pred[i] == 1 else 'Погиб'
    print(f'  P(погиб)={p_died:.2f}  P(выжил)={p_survived:.2f}  Предсказание: {pred}  Реально: {actual}')

---
## Шаг 8. Сравнение всех моделей


In [ ]:
# ============================================================
# ШАГ 8: Сравнение моделей
# ============================================================

models = {
    f'KNN (k={best_k})': (knn_best, X_test_scaled),
    f'Decision Tree (d=4)': (tree, X_test),
    'Random Forest': (forest, X_test),
    'SVM (RBF)': (svm, X_test_scaled),
    'Logistic Regression': (lr, X_test_scaled),
}

results = []
for name, (model, X_eval) in models.items():
    y_pred = model.predict(X_eval)
    acc = accuracy_score(y_test, y_pred)
    results.append({'Модель': name, 'Accuracy': acc})

results_df = pd.DataFrame(results).sort_values('Accuracy', ascending=False)

# Визуализация
fig, ax = plt.subplots(figsize=(12, 5))
colors = sns.color_palette('viridis', len(results_df))
bars = ax.barh(results_df['Модель'], results_df['Accuracy'], color=colors, edgecolor='white')

for bar, acc in zip(bars, results_df['Accuracy']):
    ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2,
           f'{acc:.2%}', va='center', fontsize=11, fontweight='bold')

ax.set_xlim(0.7, 0.9)
ax.set_title('Сравнение алгоритмов на Titanic', fontsize=14, fontweight='bold')
ax.set_xlabel('Accuracy')
plt.tight_layout()
plt.show()

print(results_df.to_string(index=False))
print()
print('Вывод: Random Forest и Logistic Regression — лучшие на этом датасете!')
print('Но разница небольшая — Titanic — простая задача.')

---
## Шаг 9. Cross-validation и GridSearchCV

### Проблема train/test split

Один split — это случайно. Можем получить «лёгкий» или «трудный» тест.

### Решение: Cross-validation

```
Данные: [1][2][3][4][5]

Фолд 1: [TEST][TRAIN][TRAIN][TRAIN][TRAIN] → score₁
Фолд 2: [TRAIN][TEST][TRAIN][TRAIN][TRAIN] → score₂
Фолд 3: [TRAIN][TRAIN][TEST][TRAIN][TRAIN] → score₃
Фолд 4: [TRAIN][TRAIN][TRAIN][TEST][TRAIN] → score₄
Фолд 5: [TRAIN][TRAIN][TRAIN][TRAIN][TEST] → score₅

Итог: mean(score₁...₅) ± std(score₁...₅)
```

Каждый пример — и в обучении, и в тесте (но не одновременно)!


In [ ]:
# ============================================================
# ШАГ 9: Cross-validation и GridSearchCV
# ============================================================

# ---- Cross-validation для Random Forest ----
rf_cv = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42)
cv_scores = cross_val_score(rf_cv, X, y, cv=5, scoring='accuracy')

print(f'Cross-validation (5 фолдов):')
print(f'  Скоры по фолдам: {cv_scores.round(3)}')
print(f'  Среднее: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}')
print()

# ---- GridSearchCV — автоматический подбор гиперпараметров ----
print('GridSearchCV: ищем лучшие параметры для Random Forest...')
print()

param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [4, 6, 8, None],
    'min_samples_split': [2, 5, 10],
}

grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=5,           # 5-fold cross-validation
    scoring='accuracy',
    n_jobs=-1,      # использовать все ядра
    verbose=0
)

grid_search.fit(X, y)

print(f'Лучшие параметры: {grid_search.best_params_}')
print(f'Лучший CV score: {grid_search.best_score_:.3f}')
print()

# Топ-5 комбинаций
results_gs = pd.DataFrame(grid_search.cv_results_)
top5 = results_gs.nsmallest(5, 'rank_test_score')[
    ['params', 'mean_test_score', 'std_test_score', 'rank_test_score']
]
print('Топ-5 комбинаций:')
for _, row in top5.iterrows():
    print(f'  #{int(row["rank_test_score"])}: {row["mean_test_score"]:.3f} ± {row["std_test_score"]:.3f} | {row["params"]}')

---
## Шаг 10. Практические задания


In [ ]:
# ============================================================
# ИНТЕРАКТИВНЫЙ: Задания
# ============================================================

tasks = widgets.Accordion(children=[
    widgets.HTML('''
        <h4>🎯 Задание 1: Gradient Boosting</h4>
        <p>Попробуйте <code>GradientBoostingClassifier</code> и <code>XGBoost</code>:</p>
        <pre>
from sklearn.ensemble import GradientBoostingClassifier
gb = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3)
gb.fit(X_train, y_train)
print(f'GB accuracy: {gb.score(X_test, y_test):.2%}')
        </pre>
        <p>Сравните с Random Forest. Кто лучше?</p>
    '''),
    widgets.HTML('''
        <h4>🎯 Задание 2: Другой датасет</h4>
        <p>Примените все 5 алгоритмов к <code>breast_cancer</code> датасету:</p>
        <pre>
from sklearn.datasets import load_breast_cancer
data = load_breast_cancer()
X_bc, y_bc = data.data, data.target
        </pre>
        <p>Какой алгоритм лучше? Почему?</p>
    '''),
    widgets.HTML('''
        <h4>🎯 Задание 3: Визуализация границ решений</h4>
        <p>Возьмите 2 признака (age, fare) и визуализируйте границы решений:</p>
        <pre>
from sklearn.inspection import DecisionBoundaryDisplay
# Создайте модель на 2 признаках и отрисуйте границу
        </pre>
    '''),
    widgets.HTML('''
        <h4>🎯 Задание 4: Pipeline</h4>
        <p>Соберите предобработку + модель в <code>Pipeline</code>:</p>
        <pre>
from sklearn.pipeline import Pipeline
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model', RandomForestClassifier())
])
pipe.fit(X_train, y_train)
        </pre>
        <p>Зачем? Чтобы не «утечь» данные из теста в обучение!</p>
    '''),
    widgets.HTML('''
        <h4>🎯 Задание 5: Предсказание с порогом</h4>
        <p>Измените порог решения с 0.5 на другое:</p>
        <pre>
probs = lr.predict_proba(X_test_scaled)[:, 1]
y_pred_custom = (probs > 0.3).astype(int)  # порог 0.3 вместо 0.5
        </pre>
        <p>Как меняется precision/recall? Когда нужен низкий порог?</p>
    '''),
])

for i, title in enumerate(['Gradient Boosting', 'Другой датасет', 'Границы решений', 'Pipeline', 'Порог решения']):
    tasks.set_title(i, f'🎯 Задание {i+1}: {title}')

display(tasks)

---
## Что вы узнали

| Алгоритм | Когда использовать | Плюсы | Минусы |
|----------|-------------------|-------|--------|
| **KNN** | Мало данных, простая задача | Простой, нет обучения | Медленный, чувствителен к масштабу |
| **Decision Tree** | Нужно объяснить решение | Понятный, не нужен масштаб | Переобучается, нестабильный |
| **Random Forest** | Табличные данные (дефолт!) | Мощный, стабильный, важность признаков | Медленнее, сложнее интерпретировать |
| **SVM** | Мало данных, много признаков | Мощный с kernel, работает в больших размерностях | Медленный, чувствителен к масштабу |
| **Logistic Regression** | Нужны вероятности, линейная задача | Быстрый, интерпретируемый | Только линейная граница |

### Правило большого пальца

```
Табличные данные?     → Random Forest или Gradient Boosting
Нужно объяснение?     → Decision Tree или Logistic Regression
Мало данных?          → SVM или KNN
Нейросеть?            → Только если данные: изображения, текст, звук
```

> Следующий блокнот: **Оценка моделей** — как правильно измерять качество и не обманывать себя.
